In [2]:
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, Qwen2VLForConditionalGeneration
import torch

# MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
MODEL_ID = "Qwen/Qwen2-VL-2B"
processor = AutoProcessor.from_pretrained(MODEL_ID)
# model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID)
model = Qwen2VLForConditionalGeneration.from_pretrained(MODEL_ID)


# 비전 인코더만 가져오기
vision_encoder = model.model.visual
# vision_encoder = torch.nn.DataParallel(model.model.visual).eval().cuda()

vision_encoder.eval().cuda()


Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.53s/it]


Qwen2VisionTransformerPretrainedModel(
  (patch_embed): PatchEmbed(
    (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
  )
  (rotary_pos_emb): VisionRotaryEmbedding()
  (blocks): ModuleList(
    (0-31): 32 x Qwen2VLVisionBlock(
      (norm1): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
      (norm2): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
      (attn): VisionSdpaAttention(
        (qkv): Linear(in_features=1280, out_features=3840, bias=True)
        (proj): Linear(in_features=1280, out_features=1280, bias=True)
      )
      (mlp): VisionMlp(
        (fc1): Linear(in_features=1280, out_features=5120, bias=True)
        (act): QuickGELUActivation()
        (fc2): Linear(in_features=5120, out_features=1280, bias=True)
      )
    )
  )
  (merger): PatchMerger(
    (ln_q): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
    (mlp): Sequential(
      (0): Linear(in_features=5120, out_features=5120, bias=True)
      (1): GEL

In [3]:
from PIL import Image
import requests
import torch

# 예시 이미지 불러오기
img = Image.open(requests.get("https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg", stream=True).raw)

# 이미지 전처리 (output: pixel_values, grid_thw)
processed = processor.image_processor(images=img, return_tensors="pt")
pixel_values = processed["pixel_values"].cuda()        # shape: (1, 3, D, H, W)
grid_thw     = processed["image_grid_thw"].cuda()      # shape: (1, 3)


In [4]:
pixel_values.shape

torch.Size([14308, 1176])

In [5]:
grid_thw.shape

torch.Size([1, 3])

In [6]:
with torch.no_grad():
    out = vision_encoder(pixel_values, grid_thw)  # shape: (tokens, dim=2048)


In [7]:
out.shape

torch.Size([3577, 1536])

In [10]:
vision_encoder.config.hidden_size
vision_encoder.config.out_hidden_size

1536